# FreshCheck Report Analysis + XAI Runner

??? notebook ?????????????????????????/????????????????????????????????:

- ????? mean ? SD ??????? seed
- ?? EDA ??? self-collected Thai retail pork images
- ????? prediction table ????? true/pred/probability
- ?? Grad-CAM ?????? EfficientNet-B0 structured multi-view
- ?? occlusion sensitivity heatmap ?????? Swin-T structured multi-view

Runtime ?????: Python 3 + A100 GPU + High-RAM. ??????????? EDA ??? CPU ??? ??? XAI ?????? GPU.


In [ ]:
#@title 1) Mount Drive and clone/update repo
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO_URL = 'https://github.com/techasit239/DADS7202_PigPicture.git'
REPO_DIR = Path('/content/DADS7202_PigPicture')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
print('repo:', REPO_DIR)


In [ ]:
#@title 2) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q seaborn opencv-python-headless tabulate


In [ ]:
#@title 3) Configure paths
from pathlib import Path
import json, math, os, random, shutil
import numpy as np
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')
ARTIFACTS = DRIVE_ROOT / 'artifacts'
REPORT_ROOT = ARTIFACTS / 'report_outputs'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CSV = ARTIFACTS / 'target_split' / 'target_holdout.csv'
SOURCE_HOLDOUT_CSV = ARTIFACTS / 'experiments_auto' / 'source_holdout.csv'

CLASS_NAMES = ['FRESH', 'HALF_FRESH', 'SPOILED']
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i:c for c,i in CLASS_TO_IDX.items()}

# ???? path ???????????? Drive ???????????????????????????????
EXPERIMENTS = {
    'efficientnet_b0_structured': {
        'model': 'efficientnet_b0', 'dropout': 0.25, 'img_size': 224, 'num_views': 3, 'best_seed': 'seed_62',
        'root': ARTIFACTS / 'exp_effb0_multiview_runs' / 'efficientnet_b0',
        'target_summary': ARTIFACTS / 'exp_effb0_multiview_runs' / '_summary' / 'efficientnet_b0_target' / 'efficientnet_b0_seed_summary.json',
        'source_summary': ARTIFACTS / 'exp_effb0_multiview_runs' / '_summary' / 'efficientnet_b0_source' / 'efficientnet_b0_seed_summary.json',
    },
    'swin_t_structured': {
        'model': 'swin_t', 'dropout': 0.30, 'img_size': 224, 'num_views': 3, 'best_seed': 'seed_52',
        'root': ARTIFACTS / 'exp_swin_multiview_runs' / 'swin_t',
        'target_summary': ARTIFACTS / 'exp_swin_multiview_runs' / '_summary' / 'swin_t_target' / 'swin_t_seed_summary.json',
        'source_summary': ARTIFACTS / 'exp_swin_multiview_runs' / '_summary' / 'swin_t_source' / 'swin_t_seed_summary.json',
    },
}

for cfg in EXPERIMENTS.values():
    cfg['checkpoint'] = cfg['root'] / cfg['best_seed'] / 'train' / 'checkpoints' / f"{cfg['model']}_best.pt"

for name, cfg in EXPERIMENTS.items():
    print('
', name)
    print('checkpoint:', cfg['checkpoint'], cfg['checkpoint'].exists())
    print('target_summary:', cfg['target_summary'], cfg['target_summary'].exists())
    print('source_summary:', cfg['source_summary'], cfg['source_summary'].exists())
print('
TARGET_CSV:', TARGET_CSV, TARGET_CSV.exists())
print('REPORT_ROOT:', REPORT_ROOT)


In [ ]:
#@title 4) Aggregate mean ? SD result tables

def extract_metric(summary, metric):
    value = summary.get(metric)
    if isinstance(value, dict):
        return value.get('mean'), value.get('std'), value.get('min'), value.get('max')
    return value, None, None, None

def read_summary(exp_name, eval_set, cfg, path):
    path = Path(path)
    if not path.exists():
        return None
    data = json.loads(path.read_text())
    row = {'experiment': exp_name, 'eval_set': eval_set, 'model': cfg['model'], 'n_runs': data.get('n_runs', data.get('runs'))}
    for metric in ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'loss']:
        mean, std, mn, mx = extract_metric(data, metric)
        row[f'{metric}_mean'] = mean
        row[f'{metric}_std'] = std
        row[f'{metric}_min'] = mn
        row[f'{metric}_max'] = mx
    return row

rows=[]
for exp_name, cfg in EXPERIMENTS.items():
    for eval_set, key in [('target_holdout','target_summary'), ('source_holdout','source_summary')]:
        row = read_summary(exp_name, eval_set, cfg, cfg[key])
        if row: rows.append(row)
summary_df = pd.DataFrame(rows)
summary_csv = REPORT_ROOT / 'structured_multiview_mean_sd_summary.csv'
summary_md = REPORT_ROOT / 'structured_multiview_mean_sd_summary.md'
summary_df.to_csv(summary_csv, index=False)
summary_md.write_text(summary_df.to_markdown(index=False), encoding='utf-8')
display(summary_df)
print('saved:', summary_csv)
print('saved:', summary_md)


In [ ]:
#@title 5) EDA for self-collected target dataset
from PIL import Image, ImageStat
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
CLASS_FOLDER_MAP = {'FR':'FRESH','FRESH':'FRESH','HF':'HALF_FRESH','HALF_FRESH':'HALF_FRESH','SP':'SPOILED','SPOILED':'SPOILED'}
TARGET_DATA_DIRS = [DRIVE_ROOT/'data'/'new_test', DRIVE_ROOT/'thai_test']

def infer_class(path):
    parts = [p.upper() for p in Path(path).parts]
    for folder, cls in CLASS_FOLDER_MAP.items():
        if folder.upper() in parts:
            return cls
    return None

def collect_images(root_dirs):
    rows=[]
    for root in root_dirs:
        if not root.exists():
            print('missing folder:', root)
            continue
        for p in root.rglob('*'):
            if p.suffix.lower() in IMAGE_EXTS:
                cls = infer_class(p)
                if cls:
                    rows.append({'path': str(p), 'filename': p.name, 'class': cls, 'dataset_root': str(root)})
    return pd.DataFrame(rows).drop_duplicates('path') if rows else pd.DataFrame()

input_df = collect_images(TARGET_DATA_DIRS)
if input_df.empty and TARGET_CSV.exists():
    input_df = pd.read_csv(TARGET_CSV)
    input_df['filename'] = input_df['path'].map(lambda x: Path(x).name)
    input_df['dataset_root'] = 'target_holdout_csv'

stats=[]
for _, row in input_df.iterrows():
    p = Path(row['path'])
    if not p.exists():
        continue
    img = Image.open(p).convert('RGB')
    gray = np.asarray(img.convert('L'))
    arr = np.asarray(img)
    stat = ImageStat.Stat(img)
    w,h = img.size
    sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if HAS_CV2 else np.nan
    stats.append({
        'path': str(p), 'filename': p.name, 'class': row['class'], 'dataset_root': row.get('dataset_root',''),
        'width': w, 'height': h, 'aspect_ratio': w/h if h else np.nan, 'megapixels': w*h/1e6,
        'brightness_mean': float(gray.mean()), 'brightness_std': float(gray.std()),
        'r_mean': stat.mean[0], 'g_mean': stat.mean[1], 'b_mean': stat.mean[2],
        'sharpness_laplacian_var': sharp,
    })

eda_df = pd.DataFrame(stats)
EDA_DIR = REPORT_ROOT / 'eda_figures'
EDA_DIR.mkdir(parents=True, exist_ok=True)
eda_csv = REPORT_ROOT / 'eda_self_collected_image_stats.csv'
eda_df.to_csv(eda_csv, index=False)
print('rows:', len(eda_df), 'saved:', eda_csv)
display(eda_df['class'].value_counts())
display(eda_df.head())


In [ ]:
#@title 6) EDA plots and sample grids
if len(eda_df) == 0:
    raise ValueError('No EDA images found. Check TARGET_DATA_DIRS or TARGET_CSV.')

sns.set_theme(style='whitegrid')

plt.figure(figsize=(6,4))
sns.countplot(data=eda_df, x='class', order=CLASS_NAMES)
plt.title('Class distribution: self-collected target images')
plt.tight_layout(); plt.savefig(EDA_DIR/'class_distribution.png', dpi=220); plt.show()

plt.figure(figsize=(6,4))
sns.scatterplot(data=eda_df, x='width', y='height', hue='class', hue_order=CLASS_NAMES)
plt.title('Image size distribution')
plt.tight_layout(); plt.savefig(EDA_DIR/'image_size_scatter.png', dpi=220); plt.show()

plt.figure(figsize=(7,4))
sns.histplot(data=eda_df, x='brightness_mean', hue='class', hue_order=CLASS_NAMES, bins=20, kde=True, element='step')
plt.title('Brightness distribution')
plt.tight_layout(); plt.savefig(EDA_DIR/'brightness_distribution.png', dpi=220); plt.show()

if eda_df['sharpness_laplacian_var'].notna().any():
    plt.figure(figsize=(7,4))
    sns.boxplot(data=eda_df, x='class', y='sharpness_laplacian_var', order=CLASS_NAMES)
    plt.title('Sharpness distribution')
    plt.tight_layout(); plt.savefig(EDA_DIR/'sharpness_boxplot.png', dpi=220); plt.show()

ncols = 6
fig, axes = plt.subplots(len(CLASS_NAMES), ncols, figsize=(2.2*ncols, 2.4*len(CLASS_NAMES)))
for r, cls in enumerate(CLASS_NAMES):
    sample = eda_df[eda_df['class']==cls].sample(min(ncols, (eda_df['class']==cls).sum()), random_state=42)
    for c in range(ncols):
        ax = axes[r,c]
        ax.axis('off')
        if c < len(sample):
            img = Image.open(sample.iloc[c]['path']).convert('RGB')
            ax.imshow(img)
            ax.set_title(cls, fontsize=9)
plt.tight_layout(); plt.savefig(EDA_DIR/'sample_grid_by_class.png', dpi=220); plt.show()
print('saved EDA figures:', EDA_DIR)


In [ ]:
#@title 7) Model helpers and structured multi-view predictions
import torch
import torch.nn.functional as F
from torchvision import transforms
from tqdm.auto import tqdm
from freshcheck.models import build_model
from freshcheck.data import sample_structured_views

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

def get_eval_transform(img_size=224):
    return transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
    ])

def load_trained_model(model_name, ckpt_path, dropout):
    model = build_model(model_name, pretrained=False, dropout=dropout).to(DEVICE)
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state)
    model.eval()
    return model

@torch.no_grad()
def predict_structured_csv(model, csv_path, img_size=224, num_views=3):
    df = pd.read_csv(csv_path)
    tfm = get_eval_transform(img_size)
    rows=[]
    for _, row in tqdm(df.iterrows(), total=len(df), desc='predict'):
        path = Path(row['path'])
        if not path.exists(): continue
        image = Image.open(path).convert('RGB')
        views = sample_structured_views(image, patch_size=img_size, num_views=num_views)
        x = torch.stack([tfm(v) for v in views]).to(DEVICE)
        logits = model(x).mean(0, keepdim=True)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred_idx = int(np.argmax(probs))
        out = {'path': str(path), 'filename': path.name, 'true_class': row.get('class',''), 'pred_class': IDX_TO_CLASS[pred_idx], 'confidence': float(probs[pred_idx])}
        for i, cls in enumerate(CLASS_NAMES): out[f'prob_{cls}'] = float(probs[i])
        rows.append(out)
    return pd.DataFrame(rows)

PRED_DIR = REPORT_ROOT/'predictions'; PRED_DIR.mkdir(parents=True, exist_ok=True)
prediction_tables = {}
for exp_name, cfg in EXPERIMENTS.items():
    if not cfg['checkpoint'].exists():
        print('skip missing checkpoint:', cfg['checkpoint']); continue
    model = load_trained_model(cfg['model'], cfg['checkpoint'], cfg['dropout'])
    pred = predict_structured_csv(model, TARGET_CSV, cfg['img_size'], cfg['num_views'])
    pred.to_csv(PRED_DIR/f'{exp_name}_target_predictions.csv', index=False)
    prediction_tables[exp_name] = pred
    print('
', exp_name)
    display(pd.crosstab(pred['true_class'], pred['pred_class']))
    display(pred.head())
    del model
    torch.cuda.empty_cache()
print('saved predictions:', PRED_DIR)


In [ ]:
#@title 8) Grad-CAM: EfficientNet-B0 structured multi-view
from matplotlib import cm
import matplotlib.pyplot as plt

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._fwd)
        self.h2 = target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.activations = o.detach()
    def _bwd(self, m, gi, go): self.gradients = go[0].detach()
    def remove(self): self.h1.remove(); self.h2.remove()
    def __call__(self, x, class_idx):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        logits[:, class_idx].sum().backward()
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1))
        cam = F.interpolate(cam[:,None,:,:], size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
        cam = cam - cam.flatten(1).min(dim=1)[0].view(-1,1,1)
        cam = cam / (cam.flatten(1).max(dim=1)[0].view(-1,1,1) + 1e-8)
        return cam.detach().cpu().numpy()

def overlay_cam(pil_img, cam, alpha=0.4):
    img = pil_img.resize((cam.shape[1], cam.shape[0]))
    img_np = np.asarray(img).astype(np.float32)/255.0
    heat = cm.get_cmap('jet')(cam)[...,:3]
    return np.clip((1-alpha)*img_np + alpha*heat, 0, 1)

def select_xai_rows(pred_df, max_misclassified=30, correct_per_class=3):
    parts=[]
    mis = pred_df[pred_df.true_class != pred_df.pred_class].copy(); mis['selection_reason']='misclassified'; parts.append(mis.head(max_misclassified))
    ok = pred_df[pred_df.true_class == pred_df.pred_class].copy()
    if len(ok):
        ok = ok.groupby('true_class', group_keys=False).apply(lambda x: x.sample(min(correct_per_class, len(x)), random_state=42))
        ok['selection_reason']='correct_sample'; parts.append(ok)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

exp_name='efficientnet_b0_structured'
cfg=EXPERIMENTS[exp_name]
pred_df=prediction_tables.get(exp_name)
if pred_df is None:
    pred_df=pd.read_csv(PRED_DIR/f'{exp_name}_target_predictions.csv')
selected=select_xai_rows(pred_df)
out_dir=REPORT_ROOT/'gradcam_effb0_structured'; out_dir.mkdir(parents=True, exist_ok=True)
model=load_trained_model(cfg['model'], cfg['checkpoint'], cfg['dropout'])
cam_runner=GradCAM(model, model.features[-1])
tfm=get_eval_transform(cfg['img_size'])
index=[]
for i,row in tqdm(selected.iterrows(), total=len(selected), desc='Grad-CAM'):
    path=Path(row['path']); image=Image.open(path).convert('RGB')
    views=sample_structured_views(image, patch_size=cfg['img_size'], num_views=cfg['num_views'])
    pred_idx=CLASS_TO_IDX[row['pred_class']]
    fig,axes=plt.subplots(1,len(views)+1,figsize=(4*(len(views)+1),4))
    axes[0].imshow(image); axes[0].axis('off'); axes[0].set_title(f"Original
true={row['true_class']} pred={row['pred_class']}
conf={row['confidence']:.3f}")
    for j,v in enumerate(views):
        x=tfm(v).unsqueeze(0).to(DEVICE)
        cam=cam_runner(x,pred_idx)[0]
        axes[j+1].imshow(overlay_cam(v,cam)); axes[j+1].axis('off'); axes[j+1].set_title(f'view {j+1}')
    plt.tight_layout()
    out_path=out_dir/f"{i:03d}_{row['selection_reason']}_{row['true_class']}_to_{row['pred_class']}_{path.stem}.png"
    plt.savefig(out_path,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
    index.append({**row.to_dict(), 'panel_path': str(out_path)})
cam_runner.remove(); del model; torch.cuda.empty_cache()
idx=pd.DataFrame(index); idx.to_csv(out_dir/'gradcam_index.csv', index=False)
print('saved:', out_dir/'gradcam_index.csv')
print('panels:', out_dir)


In [ ]:
#@title 9) Swin-T occlusion sensitivity heatmaps
# ??????? XAI ?????? transformer: ???????????????????????? confidence ??????????
@torch.no_grad()
def image_level_probs(model, views, img_size):
    tfm=get_eval_transform(img_size)
    x=torch.stack([tfm(v) for v in views]).to(DEVICE)
    logits=model(x).mean(0, keepdim=True)
    return F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

@torch.no_grad()
def occlusion_heatmaps(model, image, class_idx, img_size=224, num_views=3, grid=7):
    views=sample_structured_views(image, patch_size=img_size, num_views=num_views)
    base=image_level_probs(model, views, img_size)[class_idx]
    maps=[]
    step=math.ceil(img_size/grid)
    for vi,view in enumerate(views):
        heat=np.zeros((grid,grid), dtype=np.float32)
        for gy in range(grid):
            for gx in range(grid):
                occ=[v.copy().resize((img_size,img_size)) for v in views]
                arr=np.asarray(occ[vi]).copy()
                y0,y1=gy*step,min((gy+1)*step,img_size); x0,x1=gx*step,min((gx+1)*step,img_size)
                arr[y0:y1,x0:x1,:]=128
                occ[vi]=Image.fromarray(arr)
                heat[gy,gx]=max(0.0, float(base-image_level_probs(model, occ, img_size)[class_idx]))
        heat=heat/(heat.max()+1e-8)
        maps.append(heat)
    return views,maps,float(base)

def overlay_heat(pil_img, heat, alpha=0.45):
    heat_img=Image.fromarray((heat*255).astype(np.uint8)).resize(pil_img.size, Image.Resampling.BILINEAR)
    h=np.asarray(heat_img).astype(np.float32)/255.0
    img=np.asarray(pil_img).astype(np.float32)/255.0
    return np.clip((1-alpha)*img + alpha*cm.get_cmap('magma')(h)[...,:3],0,1)

exp_name='swin_t_structured'
cfg=EXPERIMENTS[exp_name]
pred_df=prediction_tables.get(exp_name)
if pred_df is None:
    pred_df=pd.read_csv(PRED_DIR/f'{exp_name}_target_predictions.csv')
selected=select_xai_rows(pred_df, max_misclassified=20, correct_per_class=2)
out_dir=REPORT_ROOT/'occlusion_swin_t_structured'; out_dir.mkdir(parents=True, exist_ok=True)
model=load_trained_model(cfg['model'], cfg['checkpoint'], cfg['dropout'])
index=[]
for i,row in tqdm(selected.iterrows(), total=len(selected), desc='Swin occlusion'):
    path=Path(row['path']); image=Image.open(path).convert('RGB')
    views,heats,base=occlusion_heatmaps(model, image, CLASS_TO_IDX[row['pred_class']], cfg['img_size'], cfg['num_views'], grid=7)
    fig,axes=plt.subplots(1,len(views)+1,figsize=(4*(len(views)+1),4))
    axes[0].imshow(image); axes[0].axis('off'); axes[0].set_title(f"Original
true={row['true_class']} pred={row['pred_class']}
conf={row['confidence']:.3f}")
    for j,(v,h) in enumerate(zip(views,heats)):
        axes[j+1].imshow(overlay_heat(v.resize((cfg['img_size'],cfg['img_size'])),h)); axes[j+1].axis('off'); axes[j+1].set_title(f'view {j+1}')
    plt.tight_layout()
    out_path=out_dir/f"{i:03d}_{row['selection_reason']}_{row['true_class']}_to_{row['pred_class']}_{path.stem}.png"
    plt.savefig(out_path,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
    index.append({**row.to_dict(), 'panel_path': str(out_path)})
del model; torch.cuda.empty_cache()
idx=pd.DataFrame(index); idx.to_csv(out_dir/'occlusion_index.csv', index=False)
print('saved:', out_dir/'occlusion_index.csv')
print('panels:', out_dir)


In [ ]:
#@title 10) Export output inventory and paper text snippet
text = f"""
# FreshCheck report outputs

## Tables
- Mean ? SD summary: `{REPORT_ROOT / 'structured_multiview_mean_sd_summary.csv'}`
- Mean ? SD markdown: `{REPORT_ROOT / 'structured_multiview_mean_sd_summary.md'}`

## EDA
- Image statistics CSV: `{REPORT_ROOT / 'eda_self_collected_image_stats.csv'}`
- Figures: `{REPORT_ROOT / 'eda_figures'}`

## Predictions
- Prediction tables: `{REPORT_ROOT / 'predictions'}`

## XAI
- EfficientNet-B0 Grad-CAM: `{REPORT_ROOT / 'gradcam_effb0_structured'}`
- Swin-T occlusion sensitivity: `{REPORT_ROOT / 'occlusion_swin_t_structured'}`

## Paper text suggestion
To examine whether the classifiers relied on visible pork texture rather than background or package cues, model explanation maps were generated on the target holdout set. For EfficientNet-B0, Grad-CAM was applied to the final convolutional feature block for each structured view. For Swin-T, a model-agnostic occlusion sensitivity analysis was used because the backbone is transformer-based; each spatial region was masked in turn and the decrease in predicted-class confidence was visualized. These visualizations were inspected for both correctly classified and misclassified target images.
"""
out = REPORT_ROOT / 'report_output_inventory_and_xai_text.md'
out.write_text(text, encoding='utf-8')
print(text)
print('saved:', out)
